In [7]:
import pandas as pd 


In [8]:
url = "https://klasyfikacje.stat.gov.pl/static/pkd_25/pdf/Wyjasnienia_PKD_2025.xls"
url_codes = "https://klasyfikacje.stat.gov.pl/static/pkd_25/pdf/StrukturaPKD2025.xls"
data = pd.read_excel(url, header=None, names=["code", "desc"])
codes = pd.read_excel(url_codes, names=["-", "-", "-", "code", "desc"], usecols=["code", "desc"])

In [9]:
codes.head()

,code,desc
0,Podklasa,Nazwa grupowania
1,NaN,NaN
2,NaN,"Uprawy rolne, chów i hodowla zwierząt, łowiect..."
3,NaN,Uprawy rolne inne niż wieloletnie
4,01.11.Z,"Uprawa zbóż innych niż ryż, roślin strączkowyc..."


In [10]:
codes.dropna(inplace=True)
codes.drop(0, axis=0, inplace=True)
codes = codes.reset_index(drop=True)

In [11]:
codes.code.unique().shape

(728,)

In [12]:
data.head()

,code,desc
0,SEKCJA A,"ROLNICTWO, LEŚNICTWO I RYBACTWO"
1,NaN,Sekcja ta obejmuje działalności związane z:\n-...
2,DZIAŁ 01,"Uprawy rolne, chów i hodowla zwierząt, łowiect..."
3,NaN,Dział ten obejmuje:\n- uprawę i hodowlę roślin...
4,01.1,Uprawy rolne inne niż wieloletnie


In [13]:
sum(data.code.str.contains(".Z", regex=False) == True)

602

In [14]:
df = pd.read_excel(url, header=None, names=["code", "desc"])
df['group'] = df['code'].notna().cumsum()

df = df.groupby('group').agg({
    'code': 'first',
    'desc': lambda x: ' '.join(x.dropna().astype(str))
}).reset_index(drop=True)

df = df.dropna(subset=['code'])
df['desc'] = df['desc'].str.replace(r'\s+', ' ', regex=True).str.strip()

df.head(10)

,code,desc
0,SEKCJA A,"ROLNICTWO, LEŚNICTWO I RYBACTWO Sekcja ta obej..."
1,DZIAŁ 01,"Uprawy rolne, chów i hodowla zwierząt, łowiect..."
2,01.1,Uprawy rolne inne niż wieloletnie Grupa ta obe...
3,01.11.Z,"Uprawa zbóż innych niż ryż, roślin strączkowyc..."
4,01.12.Z,Uprawa ryżu Podklasa ta obejmuje: - uprawę ryżu.
5,01.13.Z,"Uprawa warzyw, włączając melony oraz uprawa ro..."
6,01.14.Z,Uprawa trzciny cukrowej Podklasa ta nie obejmu...
7,01.15.Z,Uprawa tytoniu Podklasa ta obejmuje: - uprawę ...
8,01.16.Z,Uprawa roślin włóknistych Podklasa ta obejmuje...
9,01.19.Z,Pozostałe uprawy rolne inne niż wieloletnie Po...


In [15]:
df.shape

(1125, 2)

In [16]:
df.code = df.code[df.code.str.contains(".Z", regex=False)]
df = df.dropna(subset=["code"])
df.shape

(602, 2)

In [17]:
df.head()

,code,desc
3,01.11.Z,"Uprawa zbóż innych niż ryż, roślin strączkowyc..."
4,01.12.Z,Uprawa ryżu Podklasa ta obejmuje: - uprawę ryżu.
5,01.13.Z,"Uprawa warzyw, włączając melony oraz uprawa ro..."
6,01.14.Z,Uprawa trzciny cukrowej Podklasa ta nie obejmu...
7,01.15.Z,Uprawa tytoniu Podklasa ta obejmuje: - uprawę ...


In [18]:
missing_codes = [c for c in codes.code.unique() if c not in df.code.unique()]

In [19]:
len(missing_codes) + df.shape[0]

728

In [20]:
codes

,code,desc
0,01.11.Z,"Uprawa zbóż innych niż ryż, roślin strączkowyc..."
1,01.12.Z,Uprawa ryżu
2,01.13.Z,"Uprawa warzyw, włączając melony oraz uprawa ro..."
3,01.14.Z,Uprawa trzciny cukrowej
4,01.15.Z,Uprawa tytoniu
...,...,...
723,96.99.Z,"Pozostała działalność usługowa, gdzie indziej ..."
724,97.00.Z,Gospodarstwa domowe zatrudniające pracowników
725,98.10.Z,Gospodarstwa domowe produkujące wyroby na włas...
726,98.20.Z,Gospodarstwa domowe świadczące usługi na własn...


In [ ]:
data = 

In [25]:
data.isna().sum()

code    0
desc    0
dtype: int64

In [27]:
data.duplicated().sum()

np.int64(0)

In [32]:
data.dtypes

code    object
desc    object
dtype: object

In [61]:
data[data.code.isin(["85.59.CD"])]

,code,desc


In [2]:
from langchain_community.document_loaders import DataFrameLoader


class PKDDataImporter:
    def __init__(self):
        self.url = "https://klasyfikacje.stat.gov.pl/static/pkd_25/pdf/Wyjasnienia_PKD_2025.xls"
        self._df = None

    def _get_frame(self):
        if self._df is None:
            self._df = pd.read_excel(self.url, header=None, names=["code", "desc"])
        return self._df.copy()

    def _prepare_frame(self):
        df = self._get_frame()
        
        df['group'] = df['code'].notna().cumsum()
        df = df.groupby('group').agg({
            'code': 'first',
            'desc': lambda x: ' '.join(x.dropna().astype(str))
        }).reset_index(drop=True)
        df['desc'] = df['desc'].str.replace(r'\s+', ' ', regex=True).str.strip()
        df = df[df['code'].astype(str).str.contains(r'\.Z', na=False)]
        
        return df 

    def get_loaders(self):
        df = self._prepare_frame()
        loaders = DataFrameLoader(df, page_content_column="desc")
        return loaders.load()



/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
importer = PKDDataImporter()
loaders = importer.get_loaders()

In [4]:
loaders[0]

Document(metadata={'code': '01.11.Z'}, page_content='Uprawa zbóż innych niż ryż, roślin strączkowych i roślin oleistych na nasiona Podklasa ta obejmuje uprawę zbóż, roślin strączkowych i roślin oleistych. Ponadto obejmuje uprawę tych roślin do celów produkcji nasion. Uprawa tych zbóż jest często łączona w obrębie gospodarstwa rolnego. Podklasa ta obejmuje: - uprawę zbóż, na przykład: • pszenicy, • kukurydzy innej niż pastewnej, • sorgo, • jęczmienia, • żyta, • owsa, • prosa, • pseudozbóż, to jest owoców lub nasion, wykorzystywanych podobnie jak zboża tradycyjne, na przykład: - komosy ryżowej, - amarantusa, - chia, - uprawę roślin strączkowych, na przykład: • fasoli, • bobu, • ciecierzycy, • wspięgi chińskiej, • soczewicy, • łubinu innego niż pastewny, • grochu, • nikli indyjskiej, - uprawę roślin oleistych na nasiona, na przykład: • soi, • orzeszków ziemnych, • rącznika pospolitego, • siemienia lnianego, • gorczycy, • bawełny, • rzepaku, • szafranu, • sezamu, • słonecznika. Podklasa ta

In [ ]:
documents = []

TypeError: 'Document' object is not subscriptable

'Uprawa zbóż innych niż ryż, roślin strączkowych i roślin oleistych na nasiona Podklasa ta obejmuje uprawę zbóż, roślin strączkowych i roślin oleistych. Ponadto obejmuje uprawę tych roślin do celów produkcji nasion. Uprawa tych zbóż jest często łączona w obrębie gospodarstwa rolnego. Podklasa ta obejmuje: - uprawę zbóż, na przykład: • pszenicy, • kukurydzy innej niż pastewnej, • sorgo, • jęczmienia, • żyta, • owsa, • prosa, • pseudozbóż, to jest owoców lub nasion, wykorzystywanych podobnie jak zboża tradycyjne, na przykład: - komosy ryżowej, - amarantusa, - chia, - uprawę roślin strączkowych, na przykład: • fasoli, • bobu, • ciecierzycy, • wspięgi chińskiej, • soczewicy, • łubinu innego niż pastewny, • grochu, • nikli indyjskiej, - uprawę roślin oleistych na nasiona, na przykład: • soi, • orzeszków ziemnych, • rącznika pospolitego, • siemienia lnianego, • gorczycy, • bawełny, • rzepaku, • szafranu, • sezamu, • słonecznika. Podklasa ta nie obejmuje: - uprawy ryżu, sklasyfikowanej w 01.1

In [54]:
loaders

[Document(metadata={'code': '01.11.Z'}, page_content='Uprawa zbóż innych niż ryż, roślin strączkowych i roślin oleistych na nasiona Podklasa ta obejmuje uprawę zbóż, roślin strączkowych i roślin oleistych. Ponadto obejmuje uprawę tych roślin do celów produkcji nasion. Uprawa tych zbóż jest często łączona w obrębie gospodarstwa rolnego. Podklasa ta obejmuje: - uprawę zbóż, na przykład: • pszenicy, • kukurydzy innej niż pastewnej, • sorgo, • jęczmienia, • żyta, • owsa, • prosa, • pseudozbóż, to jest owoców lub nasion, wykorzystywanych podobnie jak zboża tradycyjne, na przykład: - komosy ryżowej, - amarantusa, - chia, - uprawę roślin strączkowych, na przykład: • fasoli, • bobu, • ciecierzycy, • wspięgi chińskiej, • soczewicy, • łubinu innego niż pastewny, • grochu, • nikli indyjskiej, - uprawę roślin oleistych na nasiona, na przykład: • soi, • orzeszków ziemnych, • rącznika pospolitego, • siemienia lnianego, • gorczycy, • bawełny, • rzepaku, • szafranu, • sezamu, • słonecznika. Podklasa t

In [28]:
df = importer._get_frame()

In [3]:
df = pd.read_excel("https://klasyfikacje.stat.gov.pl/static/pkd_25/pdf/Wyjasnienia_PKD_2025.xls", header=None, names=["code", "desc"])

In [4]:
df.head()

,code,desc
0,SEKCJA A,"ROLNICTWO, LEŚNICTWO I RYBACTWO"
1,NaN,Sekcja ta obejmuje działalności związane z:\n-...
2,DZIAŁ 01,"Uprawy rolne, chów i hodowla zwierząt, łowiect..."
3,NaN,Dział ten obejmuje:\n- uprawę i hodowlę roślin...
4,01.1,Uprawy rolne inne niż wieloletnie


In [24]:
df = importer._prepare_frame()

In [25]:
df.head()

,code,desc
3,01.11.Z,"Uprawa zbóż innych niż ryż, roślin strączkowyc..."
4,01.12.Z,Uprawa ryżu Podklasa ta obejmuje: - uprawę ryżu.
5,01.13.Z,"Uprawa warzyw, włączając melony oraz uprawa ro..."
6,01.14.Z,Uprawa trzciny cukrowej Podklasa ta nie obejmu...
7,01.15.Z,Uprawa tytoniu Podklasa ta obejmuje: - uprawę ...


In [26]:
df.shape

(602, 2)

In [17]:
len(loaders.load())

602

In [5]:
from langchain_community.document_loaders import DataFrameLoader

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
loader = DataFrameLoader(cleaned_df, page_content_column="desc")

In [19]:
cleaned_df.isna().sum()

code    0
desc    0
dtype: int64

In [27]:
loader.load()[3]

Document(metadata={'code': '01.11.Z'}, page_content='Uprawa zbóż innych niż ryż, roślin strączkowych i roślin oleistych na nasiona Podklasa ta obejmuje uprawę zbóż, roślin strączkowych i roślin oleistych. Ponadto obejmuje uprawę tych roślin do celów produkcji nasion. Uprawa tych zbóż jest często łączona w obrębie gospodarstwa rolnego. Podklasa ta obejmuje: - uprawę zbóż, na przykład: • pszenicy, • kukurydzy innej niż pastewnej, • sorgo, • jęczmienia, • żyta, • owsa, • prosa, • pseudozbóż, to jest owoców lub nasion, wykorzystywanych podobnie jak zboża tradycyjne, na przykład: - komosy ryżowej, - amarantusa, - chia, - uprawę roślin strączkowych, na przykład: • fasoli, • bobu, • ciecierzycy, • wspięgi chińskiej, • soczewicy, • łubinu innego niż pastewny, • grochu, • nikli indyjskiej, - uprawę roślin oleistych na nasiona, na przykład: • soi, • orzeszków ziemnych, • rącznika pospolitego, • siemienia lnianego, • gorczycy, • bawełny, • rzepaku, • szafranu, • sezamu, • słonecznika. Podklasa ta